# BUSI2066 · Week 4 — Cleaning, New Columns, First Charts


### Before you start

1. Open your Codespace at **github.com/codespaces** — or from your `busi2066` repository:
   green **Code** button → **Codespaces** → create one.
2. Drag **this notebook** and **`marketing_campaign.csv`** into the file list on the left. They must
   sit next to each other, or the notebook will not find the data.
3. **Select the kernel** — top right, choose the Python one. If a cell runs and nothing happens
   at all, this is why.

### Today

Last week ran out of time in the middle. Three things were on the page and never reached the room:
**cleaning a file**, **making a new column**, and **drawing a chart**. They are the three things your
own survey data will need most, so we do them properly now.

### The rule

> **AI explains. It does not write.**

Paste an error in and ask what it means. Paste a line you do not understand and ask what it does.
But type the code yourself. The one exception is marked ⭐ at the end.

---
# Part 0 · Get running again

Same as last week. `%pip` — with the percent sign — installs into the Python your notebook is
actually running, which is the whole trick.

In [ ]:
%pip install -q pandas seaborn scipy statsmodels

Now load the file. Nothing here is new — if all three cells run, you are where you finished
last week.

In [ ]:
import pandas as pd

df = pd.read_csv('marketing_campaign.csv')
df.head()

In [ ]:
print(df.shape)

`(1344, 8)` — 1,344 customers, 8 things known about each. If you got an error instead, put your
hand up now rather than in twenty minutes.

---
# Part 1 · What goes inside the brackets

You know that brackets mean **run this**. They do a second job as well: they are **where the
instructions go**. Two small examples, because both turn up in today's exercise.

## 1.1 `head(n)` — asking for a different number of rows

In [ ]:
df.head(3)

`df.head()` gave you five rows because five is what it does when you tell it nothing. Put a number
inside the brackets and it gives you that many instead.

So brackets do two jobs at once: **run this**, and **here are the instructions**. Empty brackets
just mean *run it, no instructions, use your default*.

## 1.2 `round()` — one argument or two

Every `round()` you were shown in week 2 had **two** numbers in it. It also works with one, and
today you need the one-argument version.

In [ ]:
average = df['Income'].mean()

print(average)              # the raw answer
print(round(average, 2))    # two decimal places
print(round(average))       # nearest whole number

`51466.20256...` → `51466.2` → `51466`.

The second number says **how many decimal places**. Leave it out and you get a whole number. If an
exercise asks for "the nearest whole number" and you write `round(x, 2)`, you will get `51466.2`
and think you are wrong when you are only over-precise.

---
# Part 2 · Fixing the data

You found what is wrong with this file last week: **16 missing incomes**, and a `Year_Birth` of
**1893**. Now you do something about it — which is most of what a real analyst's day looks like.

## 2.1 The sixteen missing incomes

You have three choices, and **doing nothing is also a choice** — usually the wrong one.

**Choice 1 — drop those rows.**

In [ ]:
clean = df.dropna()

print(df.shape)
print(clean.shape)

`(1344, 8)` then `(1328, 8)` — the 16 rows are gone.

⚠️ **Read that again: `df` did not change.** `dropna()` does not edit your table, it **hands you a
new one**. That is why we caught it in a variable called `clean`. If you had just run
`df.dropna()` on its own, pandas would have shown you the answer and then thrown it away.

This is true of almost every method in pandas, and it is the single most common reason a student's
"fix" appears to do nothing at all.

**Choice 2 — fill them in.** Here the obvious value is wrong, and the scale tells you why.

In [ ]:
filled = df.fillna(0)          # every missing income becomes 0

Think about what you just claimed. `Income` is a **ratio** column, so zero means *earns nothing*.
Filling with `0` tells the file that sixteen people are destitute, and every average you compute
afterwards will believe it.

The usual repair is to fill with the average of the values that are actually there.

In [ ]:
filled = df.fillna(df['Income'].mean())

**Choice 3 — leave them.** `mean()`, `max()` and `min()` skip missing values on their own, so for
simple summaries you often do not have to do anything at all.

| Choice | Code | When you would |
|---|---|---|
| Drop the rows | `df.dropna()` | Few are missing, and you can afford to lose them — 16 out of 1,344 here |
| Fill them in | `df.fillna(value)` | Many are missing, and dropping would cost too much data |
| Leave them | — | You are only summarising, and pandas skips them anyway |

There is no correct answer. There is only the one you **can defend** — and §7 of the project
guidelines says you will have to, in writing.

## 2.2 Five things that will be wrong with your own data

Missing values are the problem you can see. `info()` counts them for you. The dangerous ones are the
problems that **look like clean data** — nothing is blank, nothing errors, and the answer is quietly
wrong.

Here are the five that catch people most often, and **how you would know**. Every check below uses a
command you already have.

| What goes wrong | What it looks like | How you know |
|---|---|---|
| **Spaces on the ends** | `"Agree "` and `"Agree"` counted as two different answers | `value_counts()` shows **more categories than your question had options** |
| **Missing values that are not blank** | Somebody typed `N/A`, `-`, `none` or `999` | `value_counts()` shows the odd value sitting among the real ones |
| **A number stored as text** | One person typed `HK$1,200`, so the whole column becomes text | `info()` says `object` where you expected a number — and `describe()` **leaves the column out entirely** |
| **Impossible values** | A birth year of `1893` | `describe()` — look at **min and max first**, every time |
| **A number that is really a category** | `Response` is 0 and 1; its mean is 0.1518 | **Nothing tells you.** You have to know what the column means |

**The first four take thirty seconds.** On every new file, in this order:

1. `df.info()` — what type is each column, and how many values are actually there?
2. `df.describe()` — are the minimum and maximum possible?
3. `df['col'].value_counts()` — on every text column, does the list match the question you asked?

**The fifth one is the one to worry about**, because no command will ever flag it. `Response` has a
mean, `Kidhome` has a mean, and both means are meaningless. That is not a Python problem — it is a
*scale* problem, and it is why today's lecture spent half its time on scales.

⚠️ **In November this will be you.** Your Qualtrics export will have at least one of these, and the
exports that look cleanest are the ones worth checking hardest.

---
# Part 3 · Making a new column

Last week you worked out one person's age with `2026 - 1957`. You can do exactly the same thing to
a whole column at once, and put the answer back into the table.

In [ ]:
df['Age'] = 2026 - df['Year_Birth']

df.head()

Look at the **far right** — `Age` is now a column like any other, and `df` has nine columns instead
of eight. That is the whole pattern: **a name on the left, a calculation on the right.**

Two things worth noticing.

- **`Age` is ratio, but `Year_Birth` was interval.** Zero years old is genuinely no time; year 0
  is not. One subtraction changed the scale, and with it what you are allowed to do.
- **Your 1893 problem is now visible.** Bad data does not go away when you build on top of it —
  it comes with you.

In [ ]:
print(df['Age'].max())

**133.**

You will do this constantly in your project. A Qualtrics export gives you what you asked, not what
you want to analyse: you asked for year of birth and you want age; you asked five questions about
trust and you want one trust score. Every one of those is *a name on the left, a calculation on
the right*.

---
# Part 4 · One column, one picture

`value_counts()` gave you the education levels as a list of numbers. A chart says the same thing in
a form you can put in front of somebody.

`seaborn` is the plotting package — imported as `sns` by convention, the same way pandas is `pd`.

💡 If the next cell says `ModuleNotFoundError`, you skipped Part 0. Run `%pip install seaborn` in a
cell and try again — that is the rule from Part 0, used for real.

## 4.1 Counting categories — `countplot`

In [ ]:
import seaborn as sns

sns.countplot(data=df, y='Education')

One line, one chart.

`data=df` says which table. `y='Education'` says which column, and putting it on the **y** axis
makes the long labels read horizontally instead of overlapping.

Try `x='Education'` instead and see why we used `y`.

In [ ]:
# Your turn - swap y for x and run it

## 4.2 Spreading out a number — `histplot`

`countplot` counts how many rows fall in each category. But `Age` is not a category — it is a
number, and there are a hundred different ones. Counting each separately would tell you nothing.

A **histogram** slices the range into bins and counts how many fall in each, so you see the
**shape** of the column.

In [ ]:
sns.histplot(data=df, x='Age')

Most customers sit between about 40 and 70. And out on the right, alone, is your 133-year-old —
a **single bar miles from everything else**. That is what an outlier looks like, and it is usually
faster to see one than to calculate one.

**But `histplot` chose the number of bars for you** — 32 of them, about 3 years wide each. It will
not always choose well, and **`bins` is an argument exactly like the `3` in `head(3)`.** Part 1
again, on a chart this time.

Watch what happens when you ask for five:

In [ ]:
sns.histplot(data=df, x='Age', bins=5)

**Five bars, each covering about twenty years.** Two things went wrong.

The **shape is gone** — every customer between 30 and 92 is now squashed into three bars, and you
can no longer see where the bulk of your customers actually sit.

And the 133 is still there, but its bar now covers **112 to 133**. You can see somebody is out
there; you can no longer see *how far* out. It could be 112. It could be 133. The chart stopped
telling you.

Now the other direction:

In [ ]:
sns.histplot(data=df, x='Age', bins=60)

**Sixty bars, and now the gap is the point.** Between the oldest real customer and the 133 there
are **27 empty bins** — 27 consecutive age ranges with nobody in them. That run of emptiness is the
visual proof this is a data-entry error and not simply a very old customer.

> ⚠️ **`bins` changes what the histogram says, not just how it looks.** Every histogram you put in
> your report is a choice of bins, whether you made it or `seaborn` made it for you. **Look at two
> or three before you settle.** Next week you do this properly, alongside colours, titles and axis
> labels.

## 4.3 The rule worth remembering

You do not choose a chart by taste. **The scale of the column chooses it for you** — which is why
the scales came first.

| The column is… | Scale | Chart | Because |
|---|---|---|---|
| a category | nominal or ordinal | `sns.countplot` | you can only count how many are in each group |
| a number | interval or ratio | `sns.histplot` | the values spread along a line, so you want the shape |

Two charts is not many. Between them they cover almost every single column you will collect in your
survey. We do **two columns at a time** next week.

---
# 🛠 Exercise · Clean it, build it, draw it

Eight questions on Parts 2 to 4. **Nobody tells you which command to use** — working that out is
the exercise. Expected answers are shown where there is one, so you check by looking.

☕ **Your break is inside this block.** Work through it, then go — the sooner you are done, the
longer the break.

**1 · Make a table with no missing values, and print how many customers are in it.**

💡 Two steps, and the second one is a **built-in function**, not a method.

```
1328
```

In [ ]:
# Your code here

**2 · What is the average income in that cleaned table, to the nearest whole number?**

```
51466
```

Now run the same thing on the **original** `df`, before any cleaning. Compare the two numbers.

💡 They are the same. Work out why — it is the reason Choice 3 exists.

_(your answer)_

In [ ]:
# Your code here

**3 · Add a column called `Kids` — the total number of children at home.**

`Kidhome` is young children and `Teenhome` is teenagers. Add them together into a new column, then
count how many customers have each number.

```
1    673
0    397
2    241
3     33
```

In [ ]:
# Your code here

**4 · Draw a chart of `Marital_Status`.**

Which of the two charts, and why? Answer in words as well as code.

_(your answer)_

In [ ]:
# Your code here

**5 · Draw a chart of `Income`.**

Which one this time, and why?

_(your answer)_

In [ ]:
# Your code here

**6 · Now draw a chart of your new `Kids` column — and think before you pick.**

`Kids` is a number, so §4.3 says histogram. But there are only four possible values: 0, 1, 2 and 3.
Try **both** and look at them.

Which one is more useful here, and what does that tell you about the rule in §4.3?

_(your answer)_

In [ ]:
# Your code here - try both

**7 · Look at your income histogram from question 5. What is happening at the right-hand end?**

Say what you see, and what you would do about it.

_(your answer)_

In [ ]:
# Your code here, if you want to check a number

**8 · No code.** In your report you will have to defend what you did about the 16 missing incomes.

Which of the three choices would you take for **this** file, and what is your reason? One sentence.

_(your answer)_

---
# ⭐ Finished early? · Two things to ask AI for

**Optional and unassessed.** Two tasks. The first is the bug from §2.2, made small enough to see;
the second is the syntax you should never memorise. **Both end the same way — you check it.**

Run this. It is five answers to one survey question.

In [ ]:
answers = pd.Series(["Agree ", "Strongly agree", " Agree", "Disagree", "Agree"])

print(answers.value_counts())

Count the distinct answers a human would see: **three** — Agree, Strongly agree, Disagree.
Count what pandas reports. It is **five** — one line per answer, every count 1.

`"Agree "`, `" Agree"` and `"Agree"` are three different pieces of text as far as a computer is
concerned. This is the first row of the table in §2.2, and it is the single most common way a
survey export goes wrong: **nothing is missing, nothing errors, and your counts are simply wrong.**

Notice how you spotted it — **`value_counts()` returned five categories for a question that had
three options.** That mismatch is the whole detection method.

### Task 1 · ask for a command you have not been taught

**Now use AI properly.** Ask it:

> *"In pandas, how do I remove leading and trailing spaces from every value in a text Series?"*

Run what it gives you. Then do the thing this course is actually about — **check it**:

- Does `value_counts()` now show **three** answers instead of five, with `Agree` counted 3 times?
- Do the counts add up to 5?
- Can you explain, in one sentence, what the line it gave you does?

_(your answer)_

⚠️ **This is a place where AI may write the code for you** — because you specified exactly what it
must do, and you can check whether it did it.

In [ ]:
# Paste it here, run it, then check it

### Task 2 · ask for the thing nobody should memorise

Your `Age` histogram has no title and no axis labels. **Assignment 1 asks for both** — and this is
exactly the kind of syntax this course does not expect you to carry in your head.

So ask for it:

> *"In seaborn, how do I add a title and axis labels to a histogram?"*

Run what it gives you on the `Age` histogram. **Then check the thing that actually goes wrong.**

AI can give you working code for a label. It **cannot** know what your column means — only you know
that. So look at what it wrote:

| Label | Verdict |
|---|---|
| `Age` | It just copied the column name. Correct, and useless to a reader |
| `Age (years)` | Better — now a reader knows the unit |
| `Customer age in 2026` | **Best** — it says *what was measured, of whom, and when* |

- Does your chart have a title and both axis labels?
- **Does the y-axis label say what is being counted** — customers, not "count"?
- Would somebody who has never seen this file understand the chart from the labels alone?

_(your answer)_

**This is the §2.2 point again, from the other side.** No tool can supply what a column *means*.
That is the part of the work that stays yours.

In [ ]:
# Your Age histogram, with a title and labels


---
# ✅ Done — hand it in

**Nothing in this notebook is designed to fail**, so Run All should go top to bottom with no red.
If it stops, the error is real — read the last line first.

1. **Run All** — everything runs, no red.
2. **Save**: `Ctrl + S` (Mac: `Cmd + S`).
3. **Commit and sync** to GitHub with a message.
4. On Moodle: **this notebook file**.
5. **Stop your Codespace** — github.com/codespaces → Stop codespace.

## What you can now do

- [ ] put instructions inside brackets — `head(3)`, `round(x)` vs `round(x, 2)`
- [ ] drop or fill missing values, and **say which one you chose and why**
- [ ] know that a pandas method hands you a **new table** and leaves yours alone
- [ ] build a new column from ones you already have
- [ ] draw a `countplot` for a category and a `histplot` for a number
- [ ] pick the chart from the **scale of the column**, not from taste
- [ ] change what a chart shows with `bins`, and know it changes the *meaning*, not just the look
- [ ] ⭐ ask AI for syntax you were never taught, then **check what it gave you**

## Due Sunday, midnight

**Your brand, on Moodle, by midnight on Sunday 27 Sep.** One line: the brand, and why your team cares about it.